#Exploring how a particle wave packet interacts with a potential energy barrier

**Step 1** Save a copy of this file in your PSU google-drive.

**Step 2** In your copy, hover your mouse over the [] brackets and a "play arrow" will appear - click it to run that bit of code.

**Step 3** if you see,

```
''' type code below'''
```
written in <font color="orange">orange</font> - this is the place where you will have to add your own code using the variables explained in <font color="green">green</font>.


## Background / context

In this recitation you will explore one of the most surprising predictions of quantum mechanics: quantum tunneling. Unlike a classical particle, a quantum particle described by a wavefunction has a non-zero probability of passing through a potential-energy barrier even when its energy is less than the barrier height.

The simulations below illustrate this behavior qualitatively. Later, you will use Python to calculate the transmission probability from the analytic expressions developed in lecture and homework, allowing you to compare the exact results with the simulation.


Tunneling through a rectangular barrier: https://en.wikipedia.org/wiki/Rectangular_potential_barrier

###Physical consants:
* Speed of light: $c = 3.0 × 10^8$ m/s
* Electron rest mass energy: $m_ec^2$ = 511 keV
* $\hbar c$  = 197 eV · nm


##Computation goal

Use Python as a tool to calculate transmission probabilities for quantum tunneling and compare these results with the simulations.

#Part 1



##In the simulation below we are propagating a wave-packet towards a wall. Depending on the height of the wall the wave packet will reflect or transmit with different probabilities.


Click the "run" arrow for the simulation below

You can adjust the barrier height using the slider below. Make sure you give the simulation time to update (check the values for height and width displayed on the graph - make sure they match the sliders).

##Q1
**Q1** : Explore 3 barrier heights (in eV). The initial energy $E$ of the wave packet is 3eV. Make sure that you choose Vo close to E. Document % transmission in each case as a table (on paper). What do you observe?



In [ ]:
#This uses the split-step fourier method to propagate TDSE towards a barrier
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output # we need this trick to help with visualization (I looked it up)
import ipywidgets as widgets
from IPython.display import display

# Constants
hbarc=197 # ev nm = hbar*c
#m_e = 9.10938356e-31  # Mass of electron (kg)
m_e= 511e3 #eV/c^2
c= 3e8*1e9 #nm/s
E= 3 #eV hbarc*2*np.pi/wavelength
k0= np.sqrt(2*m_e*E)/hbarc
# Grid parameters
Lx=2000 # nm
Nx = 2024  # Number of grid points
dx = Lx / Nx  # Grid spacing
x = np.linspace(-Lx / 2, Lx / 2, Nx)

# Time parameters
dt = 3e-15  # Time step (seconds)
Nt = 2001  # Number of time steps

# Wave packet parameters
sigma_x= 40 #nm
x0= -200 #nm
kx0 = k0  # Initial wave vector

# Fourier space components
kx = np.fft.fftfreq(Nx, dx) * 2 * np.pi
K2 = kx ** 2
def run_simulation(barrier_height):
    # Barrier parameters
    bh_label=barrier_height
    barrier_height= barrier_height # eV

# Initial Gaussian wave packet
    psi0 = np.exp(-(x - x0) ** 2 / (2 * sigma_x ** 2)) * np.exp(1j * kx0 * x)
    psi0 = psi0.astype(np.complex128)

    # Normalize the initial wave packet
    psi0 /= np.sqrt(np.sum(np.abs(psi0) ** 2) * dx)

    # Potential energy (Square barrier in the center)
    V = np.zeros_like(x)
    V[x>0]=barrier_height
#V[np.abs(x) < barrier_width / 2] = barrier_height

    # Split-step Fourier method
    psi = psi0.copy()
    transmission_prob = 0

    for t in range(Nt):
        # (a) 1/2 Evolution for the potential energy in real space
        psi *= np.exp(-1j * V*c * dt / (2 * hbarc))

        # (b) Forward transform
        psi_k = np.fft.fft(psi)

        # (c) Full evolution for the kinetic energy in Fourier space
        psi_k *= np.exp(-1j * hbarc * K2 * c* dt / (2 * m_e)) # units here are not right for reduced mass needed to add c in nm/s

        # (d) Inverse Fourier transform
        psi = np.fft.ifft(psi_k)

        # (e) 1/2 Evolution for the potential energy in real space
        psi *= np.exp(-1j * V *c* dt / (2 * hbarc))

        # (f) Normalize the wavefunction after each full step (optional but good practice for numerical stability)
        psi = psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

        # Calculate transmission probability
        if t == 1000:
            Rho = np.abs(psi) ** 2
       # transmission_prob = np.sum(Rho[x > barrier_width / 2]) * dx
            transmission_prob = np.sum(Rho[x >0]) * dx
     # Visualization of wave packet evolution
        if t % 100 == 0:
            plt.figure(figsize=(8, 6))
            Rho = np.abs(psi) ** 2
            clear_output(wait=True) # Clear the output of the current graph: super important for visibility
            plt.plot(x , Rho / np.max(Rho), color='blue', lw=2)
            plt.plot(x , V / np.max(V), 'r--', lw=2)
            plt.xlim(-400, 400) # Set the x-range
            plt.title(f'Time step: {t}\n Initial Energy = {E} eV  \n [{bh_label} = barrier height:  eV] \n Transmission Probability: {transmission_prob:.3f} ')
            plt.xlabel(r'$x$ (nm)', fontsize=14)
            plt.ylabel(r'Probability Density', fontsize=14)
            plt.show()
            plt.pause(.01)  # Pause for 1 second

# Create interactive widgets
barrier_height_slider = widgets.FloatSlider(min=2.7, max=3, step=.05, value=3, description='Barrier Height (J) :')


# Link widgets to the simulation function
interactive_plot = widgets.interactive(run_simulation, barrier_height=barrier_height_slider)

# Display the widgets and the interactive plot
display(interactive_plot)

interactive(children=(FloatSlider(value=3.0, description='Barrier Height (J) :', max=3.0, min=2.7, step=0.05),…

**please note: there are some weird things that happen in the simulation above for long-times - these features are computation errors/simulation artifacts and are not realistic**

##Q2-3

**Q2** : The simulation above only approximates the transmission coefficients. Lets calculate them exactly below using the code below - which you will need to edit. Update the list of the 3 step-height values $V_{0}$ - make sure the match the ones you chose above. You will then need to type in the equations .

Recall from the homework that
$R=\frac{(k1-k2)^2}{(k1+k2)^2}$

where

$k1=\frac{\sqrt{2*m_e*E}}{ħ}$

$k2=\frac{\sqrt{2*m_e*(E-V_o)}}{ħ}$

$T=1-R$

**Q3** : Write down your 3 values of $V_{0}$ and your 3 values of T on your paper. Describe the relationship you observe between step-height and Transmission

In [ ]:
'''add your values for Vo in an array below - replace the ones with real numbers '''
Vo=np.array([1,1,1])

# The energy of the wave packet is E#
E=3
print(f'V0={Vo}')
print(f'E={E}')

''' type your equation for k1 and k2 here according to the equations above'''
# the variables to use are
# m_e   = mass of electron (in eV = mass*c^2)
# E     = the energy of the wave packet
# hbarc = hbar * c (in eV* nm)
# to use square-root type np.sqrt

k1=
k2=

''' type the quation for R the reflection coefficient here'''
# to use squared type **2
R=

print(f'R={R}')
T=1-R
print(f'T={T}')

#Part 2

##  Below we will look at a potential barrier with a fixed height and vary thickness to explore how a particle can tunnel through that barrier.

##Q4
**Q4** : Choose a barrier height and vary the width using the sliders. Document the transmission fraction for  3 situations. Write the trend you observe on paper. The initial energy of the particle hitting the barrier is 3 eV.

<font color='orange'>Sliders: remembe -changing the numbers is not instant, but slow. You can type in a new value in the slider box and click “enter” to restart simulation </font>


In [ ]:
#This uses the split-step fourier method to propagate TDSE towards a barrier
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output # we need this trick to help with visualization (I looked it up)
import ipywidgets as widgets
from IPython.display import display

# Constants
hbarc=197 # ev nm = hbar*c
#m_e = 9.10938356e-31  # Mass of electron (kg)
m_e= 511e3 #eV/c^2
c= 3e8*1e9 #nm/s
E= 3 #eV hbarc*2*np.pi/wavelength
k0= np.sqrt(2*m_e*E)/hbarc
# Grid parameters
Lx=2000 # nm
Nx = 2024  # Number of grid points
dx = Lx / Nx  # Grid spacing
x = np.linspace(-Lx / 2, Lx / 2, Nx)

# Time parameters
dt = 3e-15  # Time step (seconds)
Nt = 1501  # Number of time steps

# Wave packet parameters
sigma_x= 40 #nm
x0= -200 #nm
kx0 = k0  # Initial wave vector

# Fourier space components
kx = np.fft.fftfreq(Nx, dx) * 2 * np.pi
K2 = kx ** 2
def run_simulation(barrier_height, barrier_width):
    # Barrier parameters
    bh_label=barrier_height
    bw_label=barrier_width
    barrier_height= barrier_height # eV
    barrier_width= barrier_width # nm

# Initial Gaussian wave packet
    psi0 = np.exp(-(x - x0) ** 2 / (2 * sigma_x ** 2)) * np.exp(1j * kx0 * x)
    psi0 = psi0.astype(np.complex128)

    # Normalize the initial wave packet
    psi0 /= np.sqrt(np.sum(np.abs(psi0) ** 2) * dx)

    # Potential energy (Square barrier in the center)
    V = np.zeros_like(x)
    #V[x>0]=barrier_height
    V[np.abs(x) < barrier_width / 2] = barrier_height

    # Split-step Fourier method
    psi = psi0.copy()
    transmission_prob = 0

    for t in range(Nt):
        # (a) 1/2 Evolution for the potential energy in real space
        psi *= np.exp(-1j * V*c * dt / (2 * hbarc))

        # (b) Forward transform
        psi_k = np.fft.fft(psi)

        # (c) Full evolution for the kinetic energy in Fourier space
        psi_k *= np.exp(-1j * hbarc * K2 * c* dt / (2 * m_e)) # units here are not right for reduced mass needed to add c in nm/s

        # (d) Inverse Fourier transform
        psi = np.fft.ifft(psi_k)

        # (e) 1/2 Evolution for the potential energy in real space
        psi *= np.exp(-1j * V *c* dt / (2 * hbarc))

        # (f) Normalize the wavefunction after each full step (optional but good practice for numerical stability)
        psi = psi / np.sqrt(np.sum(np.abs(psi)**2) * dx)

        # Calculate transmission probability
        if t == 1000:
            Rho = np.abs(psi) ** 2
       # transmission_prob = np.sum(Rho[x > barrier_width / 2]) * dx
            transmission_prob = np.sum(Rho[x >0]) * dx
     # Visualization of wave packet evolution
        if t % 100 == 0:
            plt.figure(figsize=(8, 6))
            Rho = np.abs(psi) ** 2
            clear_output(wait=True) # Clear the output of the current graph: super important for visibility
            plt.plot(x , Rho / np.max(Rho), color='blue', lw=2)
            plt.plot(x , V / np.max(V), 'r--', lw=2)
            plt.xlim(-400, 400) # Set the x-range
            plt.title(f'Time step: {t}\n Initial Energy = {E} eV  \n [{bh_label} = barrier height:  eV] \n[{bw_label} = barrier width [nm]\n Transmission Probability: {transmission_prob:.3f} ')
            plt.xlabel(r'$x$ (nm)', fontsize=14)
            plt.ylabel(r'Probability Density', fontsize=14)
            plt.show()
            plt.pause(.01)  # Pause for 1 second

# Create interactive widgets
barrier_height_slider = widgets.FloatSlider(min=1, max=2.9, step=.5, value=1.5, description='Barrier Height (eV) :', readout_format='.4f')
barrier_width_slider = widgets.FloatSlider(min=1, max=5, step=2, value=3, description='Barrier Width (nm) :', readout_format='.4f')


# Link widgets to the simulation function
interactive_plot = widgets.interactive(run_simulation,
                                       barrier_width=barrier_width_slider,
                                       barrier_height=barrier_height_slider)
# Display the widgets and the interactive plot
display(interactive_plot)

interactive(children=(FloatSlider(value=3.0, description='Barrier Height (eV) :', max=5.0, min=1.0, readout_fo…

## Q5
The simulation above is qualitative - it is for illustration but not mathematically correct. It is an approximation of the real behavior. Below we calculate the transmission fraction mathematically for any barrier where Vo=7eV (E=3eV)

**Q5** : Calculate the real transmission coefficient for 3 values of barrier width. Explain the trend you observe - compare it to Q4

In [ ]:
## Below lets calculate the tunneling transmission for V0=7 eV
Vo=7
''' add your values for barrier width in an array below - replace the ones with new values'''
L=np.array([1,1,1])  # this is in nm

# The energy of the wave packet is E#
E=3
print(f'V0={Vo}')
print(f'E={E}')

# Below is the calculation for transmission through a barrier - it is done for you
alpha=np.sqrt(2*m_e*(Vo-E))/hbarc

T_tunnel= (1+ (np.sinh(alpha*L))**2/(4*E/Vo*(1-E/Vo)))**(-1)
print(T_tunnel)

# Submission instructions

## Submitting as a **PDF**
This concludes the computation part of this recitation.

To export as a PDF, in the toolbar navigate to File>>Print (or press CTRL+P), and then "print to PDF".

Submit the <font color="red">PDF</font> along with any other work (e.g. the paper copy of the recitation) to Canvas.